---
**Author:** Leonardo Gabriel Mourao Thiel  
**Project:** Master Thesis – System Inertia in the Energy System of the Future:
Model-Based Cost Optimization to Secure Inertia Requirements

**Notebook:**  Generation of Renewable Production and Inertia Profiles



**Date:** 27.04.2026  
---

# Generation of Renewable Production and Inertia Profiles

This notebook constructs time-resolved generation and inertia profiles
for renewable energy sources in the modeled 2040 power system.

The objective is to create a consistent, physically-based dataset that
captures both the temporal variability of renewable generation and
its contribution to system inertia.

---

## Background

The increasing penetration of renewable energy sources fundamentally
changes the dynamics of power systems.

Unlike conventional generation:

- wind and solar are weather-dependent  
- hydropower is driven by water inflows  
- inverter-based resources do not inherently provide rotational inertia  

To accurately represent these effects, detailed time series must be
constructed for both generation and inertia.

---

## Objective

The objective of this notebook is to:

- generate hourly production profiles for renewable technologies  
- model hydropower inflows and convert them to generation profiles  
- compute synthetic inertia contributions from wind turbines  
- harmonize all time series to a consistent temporal resolution  

---

## Methodological Approach

The analysis follows a structured pipeline:

1. Load capacity factor data for solar and wind  
2. Generate hourly production profiles using installed capacities  
3. Process hydropower inflow data and convert to hourly resolution  
4. Model synthetic inertia from wind turbines  
5. Aggregate node-level data to country level  
6. export consistent time series for further system analysis  

---

## Technologies Considered

- Solar PV (utility-scale and rooftop)  
- Wind Onshore  
- Wind Offshore  
- Hydropower (run-of-river, reservoir, pumped storage, pondage)  

---

## Key Concepts

- **Capacity factors** to represent renewable generation variability  
- **Scaling with installed capacity** to obtain real generation (MW)  
- **Synthetic inertia modeling** for wind turbines  
- **Temporal harmonization** to ensure consistency across technologies  

---

## Scope

- Target year: 2040 (based on historical proxy year)  
- Spatial resolution: market nodes aggregated to country level  
- Temporal resolution: hourly time series  

---

## Output

The notebook produces:

- hourly generation profiles for all renewable technologies  
- hourly synthetic inertia profiles for wind  
- country-level aggregated datasets  
- export-ready time series for system modeling  

---

## Interpretation Focus

The resulting time series enable:

- analysis of renewable variability  
- assessment of system dynamics  
- evaluation of inertia availability over time  

---

## Notes

This notebook provides the dynamic foundation for the subsequent
system analysis and optimization model.

All time series are constructed to ensure consistency, reproducibility,
and physical plausibility.

## 1. Environment Setup and Input Data

This section initializes the environment and loads the input data
required for generating renewable generation profiles.

The analysis focuses on solar and wind generation, using installed
capacity data and predefined system nodes.

---

### Purpose

- load capacity data for solar and wind  
- define the spatial scope (countries and nodes)  
- specify the reference year for generation profiles  

---

### Data Sources

- **Solar capacities (2040)**  
- **Wind capacities (2040)**  
- **CESA node definitions**  

---

### Key Parameters

- **target_year**: reference year for generation profiles  
- **countryList**: countries included in the analysis  
- **cesa_nodes**: spatial resolution of the system  

---

### Notes

The generation profiles are based on historical weather conditions
for the selected reference year and scaled according to installed
capacities.

In [1]:
import pandas as pd
import os
pfad = "../Data"

df_solar_capacity = pd.read_csv(pfad+"/2040/capacities_2040/solar_capacity_2040.csv", sep=';')
df_wind_capacity=  pd.read_csv(pfad+"/2040/capacities_2040/wind_cleaned.csv", sep=';')
countryList = ["AL","AT","BA","BE","BG","CH","CZ","DE","DK","ES","FR","GR"]
countryList += ["HR","HU","IT","LU","MK","ME","NL","PL","PT","RO","RS","SI","SK"]
target_year = 2016  
cesa_nodes = [
    "AL00","AT00","BE00","BEOF","BA00","BG00","CH00","CZ00","DE00","DEKF",
    "EE00","EEOF","ES00","FR00","GR00","GR03","HR00","HU00",
    "ITA00","ITCA","ITCN","ITCS","ITN1","ITS1","ITSA","ITSI",
    "LT00","LTOF","LUB1","LUF1","LUG1","LUV1","LV00",
    "ME00","MK00","NL00","NL60","NL6H","NLA0","NLBH","NLLL",
    "PL00","PT00","RO00","RS00","SI00","SK00","TR00",
    "DKW1","DKBH","DKKF","DKNS"
]



## 2. Generation of Solar Production Profiles

This section constructs hourly solar generation profiles based on
precomputed capacity factor time series.

The data is derived from the PECD dataset and represents normalized
solar generation profiles for different nodes in the system.

---

### Purpose

- generate hourly solar production profiles  
- extract data for a specific reference year  
- create a consistent dataset across all nodes  

---

### Method

- load node-specific solar capacity factor data  
- automatically identify the relevant year column  
- extract hourly values  
- combine all nodes into a unified DataFrame  

---

### Data Characteristics

- hourly resolution  
- normalized values (capacity factors)  
- node-specific generation patterns  

---

### Notes

The resulting profiles represent relative generation (0–1) and must
be scaled with installed capacities to obtain actual generation values.

In [2]:
# ---------------------------------------------------------
# Load solar generation profiles (capacity factors)
# ---------------------------------------------------------

data = {}

default_suffixes = ["_edition 2023.2"]

for country in cesa_nodes:

    found = False

    for suffix in default_suffixes:

        # Try standard solar file
        file_standard = os.path.join(
            pfad, "2040", "res_2040",
            f"PECD_LFSolarPV_2040_{country}{suffix}.csv"
        )

        # Try fallback (utility-scale)
        file_utility = os.path.join(
            pfad, "2040", "res_2040",
            f"PECD_LFSolarPVUtility_2040_{country}{suffix}.csv"
        )

        for filename in [file_standard, file_utility]:

            if os.path.exists(filename):

                df = pd.read_csv(filename, skiprows=10)

                # Find column for target year
                year_cols = [c for c in df.columns if str(target_year) in str(c)]

                if not year_cols:
                    continue

                year_col = year_cols[0]

                # Store hourly values
                data[country] = df[year_col].reset_index(drop=True)


                found = True
                break

        if found:
            break



# ---------------------------------------------------------
# Create DataFrame
# ---------------------------------------------------------

solar_df = pd.DataFrame(data)

solar_df.index += 1
solar_df.index.name = "hour"

solar_df.fillna(0, inplace=True)




## 3. Generation of Offshore Wind Profiles

This section constructs hourly offshore wind generation profiles
based on precomputed capacity factor time series.

The data is sourced from the PECD dataset and represents normalized
wind generation patterns for each node.

---

### Purpose

- generate hourly offshore wind profiles  
- extract data for the selected reference year  
- ensure consistent temporal and spatial resolution  

---

### Method

- load node-specific offshore wind datasets  
- automatically identify the target year column  
- extract hourly capacity factor values  
- combine all nodes into a unified DataFrame  

---

### Data Characteristics

- hourly resolution  
- normalized values (capacity factors)  
- node-specific wind conditions  

---

### Notes

Wind generation profiles exhibit higher variability compared to solar
and are less predictable, reflecting changing meteorological conditions.

The profiles represent relative output and must be scaled by installed
capacity to obtain actual generation.

In [3]:
# ---------------------------------------------------------
# Load offshore wind generation profiles
# ---------------------------------------------------------

data = {}

default_suffixes = ["_edition 2023.2", "_edition 2023.3"]

for country in cesa_nodes:

    found = False

    for suffix in default_suffixes:

        filename = os.path.join(
            pfad, "2040", "res_2040",
            f"PECD_Wind_Offshore_2040_{country}{suffix}.csv"
        )

        if os.path.exists(filename):

            df = pd.read_csv(filename, skiprows=10)

            # Identify column for target year
            year_cols = [c for c in df.columns if str(target_year) in str(c)]

            if not year_cols:
                continue

            year_col = year_cols[0]

            # Store values
            data[country] = df[year_col].reset_index(drop=True)


            found = True
            break



# ---------------------------------------------------------
# Create DataFrame
# ---------------------------------------------------------

wind_offshore = pd.DataFrame(data)

wind_offshore.index += 1
wind_offshore.index.name = "hour"

wind_offshore.fillna(0, inplace=True)



## 4. Generation of Onshore Wind Profiles

This section constructs hourly onshore wind generation profiles
based on precomputed capacity factor time series.

The data represents normalized wind generation patterns for each node
and complements the offshore wind and solar profiles.

---

### Purpose

- generate hourly onshore wind profiles  
- ensure full representation of wind generation  
- complete the renewable generation dataset  

---

### Method

- load node-specific onshore wind datasets  
- extract the target year column automatically  
- construct hourly capacity factor time series  
- combine all nodes into a unified DataFrame  

---

### Data Characteristics

- hourly resolution  
- normalized values (capacity factors)  
- strong temporal variability  

---

### Notes

Onshore wind typically exhibits higher variability than offshore wind
and can differ significantly across regions due to local weather conditions.

The profiles represent relative generation and must be scaled with
installed capacity to obtain actual generation values.

In [4]:
# ---------------------------------------------------------
# Load onshore wind generation profiles
# ---------------------------------------------------------

data = {}

for country in cesa_nodes:

    filename = os.path.join(
        pfad, "2040", "res_2040",
        f"PECD_Wind_Onshore_2040_{country}_edition 2023.2.csv"
    )

    if os.path.exists(filename):

        df = pd.read_csv(filename, skiprows=10)

        # Identify column for target year
        year_cols = [c for c in df.columns if str(target_year) in str(c)]

        if not year_cols:
            continue

        year_col = year_cols[0]

        data[country] = df[year_col].reset_index(drop=True)



# ---------------------------------------------------------
# Create DataFrame
# ---------------------------------------------------------

wind_onshore = pd.DataFrame(data)

wind_onshore.index += 1
wind_onshore.index.name = "hour"

wind_onshore.fillna(0, inplace=True)




## 5. Computation of Solar Generation Profiles

This section converts normalized solar capacity factor profiles into
actual generation time series by scaling them with installed capacities.

The resulting profiles are aggregated from market nodes to country level,
providing hourly solar generation for each country.

---

### Purpose

- transform capacity factors into real generation (MW)  
- incorporate installed capacities  
- aggregate node-level data to country level  
- generate consistent time series for further analysis  

---

### Method

1. filter capacity data by solar type  
2. map installed capacities to market nodes  
3. multiply capacity factors with installed capacity  
4. aggregate node-level generation to countries  
5. construct a time-indexed DataFrame  
6. export results to Excel  

---

### Key Concept

Solar generation is computed as:

\[
\text{Generation}_{t} = \text{Capacity Factor}_{t} \times \text{Installed Capacity}
\]

---

### Notes

The resulting time series represent hourly solar generation in MW
for each country and can be directly used in system analysis and modeling.

In [5]:
import pandas as pd

import pandas as pd

# Länderliste
countryList = ["AL","AT","BA","BE","BG","CH","CZ","DE","DK","ES","FR",
               "GR","HR","HU","IT","LU","MK","ME","NL","PL","PT","RO","RS","SI","SK"]

# Mapping Market Nodes → Land (CESA)
country_nodes_map = {
    "AL": ["AL00"],
    "AT": ["AT00"],
    "BA": ["BA00"],
    "BE": ["BE00","BEOF"],
    "BG": ["BG00"],
    "CH": ["CH00"],
    "CZ": ["CZ00"],
    "DE": ["DE00","DEKF"],
    "DK": ["DKBH","DKE1","DKKF","DKNS","DKW1"],
    "ES": ["ES00"],
    "FR": ["FR00"],
    "GR": ["GR00","GR03"],
    "HR": ["HR00"],
    "HU": ["HU00"],
    "IT": ["ITA00","ITCA","ITCN","ITCS","ITN1","ITS1","ITSA","ITSI"],
    "LU": ["LUB1","LUF1","LUG1","LUV1"],
    "MK": ["MK00"],
    "ME": ["ME00"],
    "NL": ["NL00","NL60","NL6H","NLA0","NLBH","NLLL"],
    "PL": ["PL00"],
    "PT": ["PT00"],
    "RO": ["RO00"],
    "RS": ["RS00"],
    "SI": ["SI00"],
    "SK": ["SK00"]
}

def generate_Solar_Production(solar_type, output_file, start_date="2040-01-01 00:00:00"):

    # -----------------------------
    # 1) Market Nodes DataFrame erstellen
    # -----------------------------
    type_df = df_solar_capacity[df_solar_capacity["solar_type"] == solar_type]
    capacity_map = type_df.set_index("market_node")["capacity"].to_dict()
    # Nur Market Nodes aus country_nodes_map, die auch in solar_df existieren
    allowed_nodes = []
    for nodes in country_nodes_map.values():
        allowed_nodes.extend(nodes)
    allowed_nodes = [n for n in allowed_nodes if n in solar_df.columns]
    
    # DataFrame mit Produktion * Kapazität
    df_nodes = pd.DataFrame(index=solar_df.index)
    for node in allowed_nodes:
        df_nodes[node] = solar_df[node] * capacity_map.get(node, 0) * 1000
    # -----------------------------
    # 2) Länder-DataFrame erstellen
    # -----------------------------
    df_country = pd.DataFrame(index=df_nodes.index)
    for country in countryList:
        nodes = [n for n in country_nodes_map[country] if n in df_nodes.columns]
        if nodes:
            df_country[country] = df_nodes[nodes].sum(axis=1)
        else:
            df_country[country] = 0

    # Zeitspalte hinzufügen
    df_country["time"] = pd.date_range(start=start_date, periods=len(df_country), freq="h")
    cols = ["time"] + [c for c in df_country.columns if c != "time"]
    df_country = df_country[cols]

    # Excel-Ausgabe
    df_country.to_excel(output_file, index=False)


# ---------------------------------------------------------
# 5) Aufrufe
# ---------------------------------------------------------
generate_Solar_Production(
    "solar_pv",
    pfad + "/2040/Generation_2040/SolarPV_2040.xlsx"
)

generate_Solar_Production(
    "solar_rooftop",
    pfad + "/2040/Generation_2040/SolarRooftop_2040.xlsx"
)


## 6. Computation of Wind Generation Profiles

This section computes hourly wind generation profiles by scaling
capacity factor time series with installed wind capacities.

The methodology is identical to solar generation but applied to
both onshore and offshore wind technologies.

---

### Purpose

- generate hourly wind generation profiles (MW)  
- distinguish between onshore and offshore wind  
- aggregate generation from market nodes to country level  

---

### Method

1. load capacity factor time series (wind profiles)  
2. map installed capacities to market nodes  
3. compute generation using scaling  
4. aggregate node-level generation to countries  
5. create time-indexed datasets  
6. export results for further analysis  

---

### Key Concept

Wind generation is computed as:

\[
\text{Generation}_{t} = \text{Capacity Factor}_{t} \times \text{Installed Capacity}
\]

---

### Notes

Both onshore and offshore wind are considered separately, as they
exhibit different temporal characteristics and capacity factors.

In [6]:
def generate_Wind_Production(
    df_factors,
    capacity_map,
    output_file,
    start_date="2040-01-01 00:00:00"
):
    """
    df_factors   : DataFrame mit Leistungsfaktoren [0..1], Spalten = Market Nodes
    capacity_map : dict {market_node: installed_capacity_GW}
    """

    # 1️⃣ Nur Market Nodes berücksichtigen, die erlaubt UND vorhanden sind
    allowed_nodes = [
        n for nodes in country_nodes_map.values() for n in nodes
        if n in df_factors.columns
    ]

    # 2️⃣ Market-Node-Produktion (MW)
    df_nodes = pd.DataFrame(index=df_factors.index)
    for node in allowed_nodes:
        df_nodes[node] = df_factors[node] * capacity_map.get(node, 0) * 1000

    # 3️⃣ Länderaggregation
    df_country = pd.DataFrame(index=df_nodes.index)
    for country in countryList:
        nodes = [n for n in country_nodes_map[country] if n in df_nodes.columns]
        df_country[country] = df_nodes[nodes].sum(axis=1) if nodes else 0

    # 4️⃣ Zeitspalte
    df_country["time"] = pd.date_range(
        start=start_date,
        periods=len(df_country),
        freq="h"
    )

    df_country = df_country[["time"] + countryList]

    # 5️⃣ Export
    df_country.to_excel(output_file, index=False)
    
capacity_map_onshore = (
    df_wind_capacity[df_wind_capacity["wind_type"] == "onshore"]
    .groupby("market_node")["installed_capacity_GW"]
    .sum()
    .to_dict()
)
capacity_map_offshore = (
    df_wind_capacity[df_wind_capacity["wind_type"] == "offshore"]
    .groupby("market_node")["installed_capacity_GW"]
    .sum()
    .to_dict()
)
generate_Wind_Production(
    df_factors=wind_onshore,
    capacity_map=capacity_map_onshore,
    output_file=pfad + "/2040/Generation_2040/Wind_Onshore_2040.xlsx"
)

generate_Wind_Production(
    df_factors=wind_offshore,
    capacity_map=capacity_map_offshore,
    output_file=pfad + "/2040/Generation_2040/Wind_Offshore_2040.xlsx"
)


## Literature Basis

The formulation of synthetic inertia implemented in this work is based on
the methodology proposed by Xu et al. (2021).

---

## Reference

Xu, B., et al. (2021).  
*Virtual Inertia Coordinated Allocation Method Considering Inertia Demand and Wind Turbine Inertia Response Capability*.  
Energies, 14(16), 5002.  
https://doi.org/10.3390/en14165002  

---

## Adaptation in This Work

The original approach is adapted to suit system-level analysis.  
In particular:

- the model is simplified for computational efficiency  
- turbine control dynamics are represented in an aggregated form  
- the formulation is integrated into a broader power system framework  

---

## Remarks

The implementation follows the conceptual structure of the original study,
but does not reproduce the full level of detail of the control strategy.

In [7]:
import math
def berechne_K(omega, omega_min, omega_n, P, P_H, P_L):
    k_omega = (omega**2 - omega_min**2) / omega_n**2
    if(k_omega<=0):
        return 0
    k_P = (P_H- P) / (P_H - P_L)
    K = k_omega * k_P
    return K


def berechne_HW_lim(P_H, P_L, df_dt=1):
    HW_lim = (P_H - P_L) / 2 * df_dt
    return HW_lim

def berechne_omega(P_aktuell, Cp, R, lambda_opt):
    """
    Berechnet die Rotorwinkelschnelle omega aus der aktuellen Leistung
    """
    A=math.pi*(R**2)
    rho=1.225
    omega = (lambda_opt / R) * (2 * P_aktuell *1e6 / (rho * A * Cp))**(1/3)
    return omega
def berechne_inertia(P_aktuell, Cp, R, lambda_opt,P_H,P_L,df_dt,omega_min, omega_n):
    """
    Berechnet die Trägheit  aus der aktuellen Leistung
    """
    omega=berechne_omega(P_aktuell, Cp, R, lambda_opt)
    HW_lim=berechne_HW_lim(P_H, P_L, df_dt)
    k=berechne_K(omega, omega_min, omega_n, P_aktuell, P_H, P_L)
    inertia=k*P_H*HW_lim
    return inertia

## Wind Turbine Assumptions

For onshore wind turbines, the Vestas V117-4.2 MW is used as the
reference model.

---

### Rationale

This turbine represents a modern onshore wind technology and provides
realistic technical parameters for:

- rated power  
- rotor radius  
- aerodynamic efficiency  
- operating range  

---

### Application

The technical parameters of the reference turbine are used to:

- compute rotor speed  
- estimate available kinetic energy  
- model synthetic inertia contribution  

---

### Remarks

Using a representative reference turbine enables a consistent and
computationally efficient approximation of wind turbine behavior
at system level.

In [8]:
# ---------------------------------------------------------
# Parameters (onshore wind turbine)
# ---------------------------------------------------------

P_H = 4.2
P_L = 0.07
Cp = 0.41
omega_min = 0.22
omega_n = 1.035
wind_low = 3
wind_high = 14
R = 58.5

lambda_opt = (R * omega_n) / wind_high
df_dt = 1


# ---------------------------------------------------------
# Compute inertia time series
# ---------------------------------------------------------

wind_onshore_inertia = pd.DataFrame(index=wind_onshore.index)

for node in wind_onshore.columns:

    inertia_series = []

    for cf in wind_onshore[node]:

        P_aktuell = cf * P_H

        inertia = berechne_inertia(
            P_aktuell, Cp, R, lambda_opt,
            P_H, P_L, df_dt, omega_min, omega_n
        ) / P_H   # normalization

        inertia_series.append(inertia)

    wind_onshore_inertia[node] = inertia_series





# ---------------------------------------------------------
# Aggregate to country level
# ---------------------------------------------------------

generate_Wind_Production(
    df_factors=wind_onshore_inertia,
    capacity_map=capacity_map_onshore,
    output_file=pfad + "/2040/Generation_2040/Wind_Onshore_Inertia_2040.xlsx"
)

## 9. Synthetic Inertia from Offshore Wind

For offshore wind turbines, the NREL 5 MW reference turbine is used
to model the inertia contribution.

---

### Rationale

Offshore wind turbines differ significantly from onshore turbines in terms of:

- higher rated power  
- larger rotor diameter  
- higher and more stable wind speeds  
- increased capacity factors  

These characteristics influence both the available kinetic energy
and the resulting synthetic inertia contribution.

---

### Method

The same methodology as for onshore wind is applied:

1. compute rotor speed based on power output  
2. estimate available kinetic energy  
3. calculate scaling factors  
4. derive time-dependent inertia contribution  

---

### Parameterization

The model uses parameters representative of the NREL 5 MW offshore turbine.

---

### Notes

Offshore wind turbines generally provide more stable and higher inertia
potential due to more consistent wind conditions and higher operating levels.

In [9]:
# ---------------------------------------------------------
# Parameters (offshore wind turbine - NREL 5 MW)
# ---------------------------------------------------------

P_H = 5
P_L = 0.09
Cp = 0.42
omega_min = 0.723
omega_n = 1.267
wind_low = 3
wind_high = 11.4
R = 63

lambda_opt = (R * omega_n) / wind_high
df_dt = 1


# ---------------------------------------------------------
# Compute inertia time series
# ---------------------------------------------------------

wind_offshore_inertia = pd.DataFrame(index=wind_offshore.index)

for node in wind_offshore.columns:

    inertia_series = []

    for cf in wind_offshore[node]:

        P_aktuell = cf * P_H

        inertia = berechne_inertia(
            P_aktuell, Cp, R, lambda_opt,
            P_H, P_L, df_dt, omega_min, omega_n
        ) / P_H  # normalization

        inertia_series.append(inertia)

    wind_offshore_inertia[node] = inertia_series




# ---------------------------------------------------------
# Aggregate to country level
# ---------------------------------------------------------

generate_Wind_Production(
    df_factors=wind_offshore_inertia,
    capacity_map=capacity_map_offshore,
    output_file=pfad + "/2040/Generation_2040/Wind_Offshore_Inertia_2040.xlsx"
)

## 10. Processing of Hydropower Inflow Profiles

This section processes hydropower inflow data and constructs
daily time series for different hydropower technologies.

Unlike wind and solar, hydropower generation is driven by water inflows,
which are provided as aggregated time series in the input data.

---

### Purpose

- generate daily inflow profiles for hydropower  
- differentiate between hydropower technologies  
- aggregate node-level data to country level  

---

### Technologies Considered

- Run-of-River  
- Reservoir  
- Pumped Storage (Open and Closed)  
- Pondage  

---

### Method

1. load node-specific inflow datasets  
2. extract values for a reference year  
3. convert weekly data to daily resolution where required  
4. handle missing or incomplete data  
5. aggregate node-level inflows to country level  
6. construct time-indexed datasets  

---

### Key Concept

Hydropower is modeled based on **inflows**, not capacity factors:

- inflows represent available water volume  
- generation depends on reservoir operation and constraints  

---

### Notes

A historical year (proxy year) is used to represent hydrological conditions,
analogous to the approach used for wind and solar profiles.

In [ ]:
# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

proxy_year = 2016
pfad_hydro = os.path.join(pfad, "2040", "res_2040")

sheets = [
    "Run of River - Year Dependent",
    "Reservoir - Year Dependent",
    "PS Open - Year Dependent",
    "PS Closed - Year Dependent",
    "Pondage - Year Dependent",
]

suffixes = ["00", "G1", ""]


# ---------------------------------------------------------
# Helper: weekly → daily conversion
# ---------------------------------------------------------

def weekly_to_daily(series_weekly):
    daily_values = []
    for v in series_weekly:
        daily_values.extend([v / 7] * 7)
    return pd.Series(daily_values[:366], dtype=float)


# ---------------------------------------------------------
# Load node-level hydro data
# ---------------------------------------------------------

hydro_nodes = {sheet: pd.DataFrame() for sheet in sheets}

for country, nodes in country_nodes_map.items():
    for node in nodes:

        file_found = False

        for suffix in suffixes:

            fname = f"PEMMDB_{node}{suffix}_Hydro_Inflows_2040.xlsx"
            fpath = os.path.join(pfad_hydro, fname)

            if not os.path.exists(fpath):
                continue

            file_found = True

            dfs = pd.read_excel(fpath, sheet_name=sheets, skiprows=1)

            for sheet_name, df in dfs.items():

                if proxy_year not in df.columns:
                    continue

                values = df[proxy_year].dropna().reset_index(drop=True)

                # Weekly → daily conversion
                if sheet_name in [
                    "Reservoir - Year Dependent",
                    "PS Open - Year Dependent",
                    "PS Closed - Year Dependent"
                ]:
                    values_daily = weekly_to_daily(values)

                else:
                    # Ensure full year coverage
                    if len(values) < 366:
                        if len(values) == 0:
                            values = pd.Series([0.0] * 366, dtype=float)
                        else:
                            values = values.reindex(range(366), fill_value=values.iloc[-1])

                    values_daily = values.astype(float)

                hydro_nodes[sheet_name][node] = values_daily

            break

# -------------------------------
# 2️⃣ Länder-Summen erstellen
# -------------------------------
hydro_country = {}

for sheet_name, df_nodes in hydro_nodes.items():
    if df_nodes.empty:
        continue

    df_country = pd.DataFrame(index=df_nodes.index)

    for country, nodes in country_nodes_map.items():
        available = [n for n in nodes if n in df_nodes.columns]
        df_country[country] = df_nodes[available].sum(axis=1)

    # Zeitspalte (Datum)
    df_country["time"] = pd.date_range(
        start=f"{proxy_year}-01-01",
        periods=len(df_country),
        freq="D"
    )

    # Spaltenreihenfolge: Zeit zuerst
    hydro_country[sheet_name] = df_country[["time"] + [c for c in df_country.columns if c != "time"]]


## 11. Conversion of Hydropower Inflows to Hourly Profiles

This section converts daily hydropower inflow data into hourly
time series to ensure consistency with the temporal resolution
of other generation sources.

---

### Purpose

- harmonize temporal resolution across all generation technologies  
- convert daily inflows into hourly values  
- prepare hydropower data for system-level modeling  

---

### Method

1. define an hourly time index for the full year  
2. repeat daily values for each hour of the day  
3. distribute daily inflows evenly across 24 hours  
4. convert units where required  
5. export hourly profiles per hydropower category  

---

### Key Assumption

Daily inflows are uniformly distributed over the day:

\[
\text{Hourly Value} = \frac{\text{Daily Value}}{24}
\]

---

### Notes

This simplification ensures consistency with hourly simulation models,
while preserving the overall daily inflow structure.

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import os
# Kategorien / Hydro Sheets
categories = [
    "Run of River - Year Dependent",
    "Reservoir - Year Dependent",
    "PS Open - Year Dependent",
    "PS Closed - Year Dependent",
    "Pondage - Year Dependent",
]
# ---------------------------------------------------------
# Convert daily hydro inflows to hourly profiles
# ---------------------------------------------------------

# Define hourly timeline (leap year: 366 days)
start_date = datetime(2040, 1, 1)
time_index = [
    start_date + timedelta(hours=i)
    for i in range(366 * 24)
]


# Output folder
output_folder = os.path.join(pfad, "2040", "Generation_2040")
os.makedirs(output_folder, exist_ok=True)


# ---------------------------------------------------------
# Loop over hydropower categories
# ---------------------------------------------------------

for category in categories:

    df_out = pd.DataFrame({"Timestamp": time_index})

    for country in country_nodes_map.keys():

        if category in hydro_country:

            df_sheet = hydro_country[category]

            if country in df_sheet.columns:

                daily_values = df_sheet[country]

                # Convert daily → hourly
                hourly_values = (
                    daily_values.repeat(24) / 24
                ).reset_index(drop=True) * 1000  # unit conversion

                df_out[country] = hourly_values

            else:
                df_out[country] = 0

        else:
            df_out[country] = 0


    # ---------------------------------------------------------
    # Export
    # ---------------------------------------------------------

    filename = (
        category
        .replace(" - Year Dependent", "")
        .replace(" ", "_")
        + ".xlsx"
    )

    filepath = os.path.join(output_folder, filename)

    df_out.to_excel(filepath, index=False)

print("Done")